# Course 6: Winning Online AI Competitions
## Session 8: NLP and LLM competitions: predicting preferences

**Goal:** predict which of two answers a person prefers, fix the position problem with swap augmentation and swap TTA, and use soft pseudo-labels.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on synthetic or built-in data. Cells labelled NOT RUN need a Kaggle account, a competition's data, a GPU or an extra library.

### 1. The task
As in the Chatbot Arena competitions: each row has a prompt, answer A and answer B, and the label says which one the person preferred, or a tie. The metric is log loss over the three classes. This notebook builds a synthetic version with a hidden quality rule so that everything runs on a CPU; the real version fine-tunes an LLM (last cell).

In [ ]:
import numpy as np, pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
rng = np.random.default_rng(0)
GOOD = ["because", "therefore", "for example", "step by step", "in summary", "specifically", "the evidence shows"]
FILL = ["the", "thing", "is", "that", "it", "really", "very", "quite", "maybe", "stuff", "like", "you know", "and so on", "basically"]
BAD = ["i am not sure", "as an ai", "cannot help", "sorry"]
def make_answer(quality):
    n = int(rng.integers(8, 40) + 15 * quality)
    words = list(rng.choice(FILL, n)) + list(rng.choice(GOOD, int(rng.poisson(1 + 3 * quality)))) + list(rng.choice(BAD, int(rng.poisson(1.5 * (1 - quality)))))
    rng.shuffle(words); return " ".join(words)
def make_data(n):
    rows = []
    for i in range(n):
        qa, qb = rng.random(), rng.random(); diff = qa - qb + rng.normal(scale=0.25)
        label = 2 if abs(diff) < 0.12 else (0 if diff > 0 else 1)                  # 0: A wins, 1: B wins, 2: tie
        rows.append({"id": i, "prompt": f"question {i}", "response_a": make_answer(qa), "response_b": make_answer(qb), "label": label})
    return pd.DataFrame(rows)
train, test, unlabeled = make_data(1500), make_data(1500), make_data(3000)
print(train.label.value_counts(normalize=True).sort_index().round(3).to_dict()); print(train.response_a[0][:90], "...")

### 2. A simple model
TF-IDF of each answer, placed side by side, plus the two lengths. The classifier has separate weights for the A slot and the B slot.

In [ ]:
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True).fit(pd.concat([train.response_a, train.response_b]))
def featurize(df, swap=False):
    a, b = (df.response_b, df.response_a) if swap else (df.response_a, df.response_b)
    lens = np.c_[a.str.split().str.len(), b.str.split().str.len()] / 50.0
    return hstack([vec.transform(a), vec.transform(b), csr_matrix(lens)]).tocsr()
SWAP_LABEL = np.array([1, 0, 2])                                                   # after swapping, 'A wins' becomes 'B wins'
def fit(X, y, sample_weight=None): return LogisticRegression(C=0.3, max_iter=2000).fit(X, y, sample_weight=sample_weight)

base = fit(featurize(train), train.label)
p_base = base.predict_proba(featurize(test)); ll_base = log_loss(test.label, p_base, labels=[0, 1, 2])
uniform = log_loss(test.label, np.full((len(test), 3), 1 / 3), labels=[0, 1, 2])
print(f"predicting 1/3 each: log loss {uniform:.4f} | baseline model: {ll_base:.4f}")

### 3. The position problem
Nothing about the task depends on which answer is called A. A model should give mirrored predictions when the two answers are swapped. Check whether ours does.

In [ ]:
p_swapped = base.predict_proba(featurize(test, swap=True))[:, SWAP_LABEL]             # predict on swapped inputs, then swap the classes back
inconsistency = np.abs(p_base - p_swapped).mean()
print(f"mean absolute difference between the two predictions for the same pair: {inconsistency:.4f}  (0 would be perfectly consistent)")

### 4. Two fixes
- **Swap augmentation (training):** add every training pair again with A and B exchanged and the label mirrored. Twice the data for free.
- **Swap TTA (prediction):** predict both orders and average. The result is exactly consistent.

In [ ]:
from scipy.sparse import vstack
X_aug = vstack([featurize(train), featurize(train, swap=True)]); y_aug = np.r_[train.label.values, SWAP_LABEL[train.label.values]]
aug = fit(X_aug, y_aug)
def predict(model, df, tta):
    p = model.predict_proba(featurize(df))
    return (p + model.predict_proba(featurize(df, swap=True))[:, SWAP_LABEL]) / 2 if tta else p
rows = {"baseline": ll_base, "baseline + swap TTA": log_loss(test.label, predict(base, test, True), labels=[0, 1, 2]),
        "swap augmentation": log_loss(test.label, predict(aug, test, False), labels=[0, 1, 2]),
        "swap augmentation + swap TTA": log_loss(test.label, predict(aug, test, True), labels=[0, 1, 2])}
for k, v in rows.items(): print(f"{k:30s} log loss {v:.4f}")
assert rows["swap augmentation + swap TTA"] < rows["baseline"] and rows["baseline + swap TTA"] < rows["baseline"]

### 5. Soft pseudo-labels
You often have many unlabeled pairs (the test set itself in some competitions, or extra data). Predict them with your best model, and train again with those predictions as *soft* labels: the full probability vector, not the top class. Hard pseudo-labels throw away the model's uncertainty and make it overconfident.

In [ ]:
def fit_soft(X_lab, y_lab, X_unl, P_unl, weight=0.5):
    """Train on labeled rows plus unlabeled rows repeated once per class, weighted by the predicted probability of that class."""
    Xs = vstack([X_lab] + [X_unl] * 3); ys = np.r_[y_lab, np.repeat([0, 1, 2], X_unl.shape[0])]
    ws = np.r_[np.ones(X_lab.shape[0]), weight * P_unl[:, 0], weight * P_unl[:, 1], weight * P_unl[:, 2]]
    return fit(Xs, ys, ws)
P_unl = predict(aug, unlabeled, True)
X_unl = vstack([featurize(unlabeled), featurize(unlabeled, swap=True)]); P_unl2 = np.vstack([P_unl, P_unl[:, SWAP_LABEL]])
soft = fit_soft(X_aug, y_aug, X_unl, P_unl2)
hard_labels = np.eye(3)[P_unl2.argmax(1)]; hard = fit_soft(X_aug, y_aug, X_unl, hard_labels)
rows["+ soft pseudo-labels"] = log_loss(test.label, predict(soft, test, True), labels=[0, 1, 2])
rows["+ hard pseudo-labels"] = log_loss(test.label, predict(hard, test, True), labels=[0, 1, 2])
for k in ["swap augmentation + swap TTA", "+ soft pseudo-labels", "+ hard pseudo-labels"]: print(f"{k:30s} log loss {rows[k]:.4f}")

Report these two lines as they come out. Pseudo-labels help most when the unlabeled set is large and the teacher is much stronger than the student; with a simple model teaching itself, the gain can be small or absent, and hard labels usually do worse than soft ones under log loss. In the real competitions the big gain came from a large teacher model labelling extra data for a smaller student.

### 6. Calibration for log loss
A last cheap step: fit one temperature on out-of-fold predictions.

In [ ]:
from scipy.optimize import minimize_scalar
from sklearn.model_selection import cross_val_predict
oof = cross_val_predict(LogisticRegression(C=0.3, max_iter=2000), featurize(train), train.label, cv=5, method="predict_proba")
def with_temperature(P, T): L = np.log(P) / T; E = np.exp(L - L.max(1, keepdims=True)); return E / E.sum(1, keepdims=True)
T_best = minimize_scalar(lambda T: log_loss(train.label, with_temperature(oof, T), labels=[0, 1, 2]), bounds=(0.3, 3), method="bounded").x
print(f"best temperature on OOF predictions: {T_best:.2f} | test log loss {ll_base:.4f} -> {log_loss(test.label, with_temperature(p_base, T_best), labels=[0, 1, 2]):.4f}")

### 7. The real approach: QLoRA on an LLM (NOT RUN here)

In [ ]:
# NOT RUN: needs GPUs and `pip install transformers peft bitsandbytes`. Winning solutions used models of several billion parameters.
# 1. Input text: "<prompt> ... <response_a> ... <response_b> ...", truncated so that both answers keep their beginning and end.
# 2. Model: AutoModelForSequenceClassification with num_labels=3 on a 4-bit base model, plus LoRA adapters (r 16 to 64) on the attention layers.
# 3. Training: 1-2 epochs, learning rate about 1e-4 for the adapters, with swap augmentation.
# 4. Inference: sort by length to batch efficiently (session 9), predict both orders, average (swap TTA).
# 5. Distillation: label extra conversations with the large model and train a smaller one on the soft labels.
# Validate with grouped folds by prompt: the same prompt must not appear on both sides.

### TODO
1. Replace the model with one that is consistent by construction: features `f(A) - f(B)` and a classifier with no intercept for the two win classes. Compare with swap TTA.
2. Add a length-only baseline. How much of the score is 'longer answers win'? What does that say about the data?
3. Vary the pseudo-label weight over 0.1, 0.5, 1.0 and the unlabeled set size. When does it help?